# N172 · Medium综合题与机制组合

**目标：** 在两个已知机制之间定义清楚接口。

**先修：** N061, N094, N097, N035。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 排序+堆；前缀+哈希；单调结构+贪心；分离状态职责。

**配套讲解来源：** [同名逐章意见](../../comment/172_medium_pattern_combinations.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章的方法论解决的问题是：**中等难度综合题往往不是"新算法"，而是两个已知机制的组合（比如"计数 + 堆"、"排序 + 贪心"）。组合的关键是在两个机制之间定义清楚接口——数据从谁流向谁、以什么格式传递**。本章特别强调：三道题看起来都像"局部选择 + 结构约束"，但不能因此共用一个没有证明过的贪心模板；每一道都要单独论证自己的选择为什么对。

三道承载题分别是：任务冷却排期（LC 621）、字符串重排（LC 767）、视频片段覆盖（LC 1024）。我们各给一个具体到数字的小例子：

- **任务冷却**：`tasks='AAABBB'`，冷却间隔 `n=2`（同种任务两次执行之间至少隔 2 个时间单位）。最优时间线是 `A B _ A B _ A B`（`_` 是强制空闲），总长 **8**。
- **字符串重排**：`s='aaabbc'`，要求重排后相邻字符都不相同。按"每次优先输出剩余最多的字符、并扣住刚输出的"策略，得到的输出是 **`ababac`**（第三节会一轮一轮手推）；而 `s='aaab'` 中 a 出现 3 次，超过 `(4+1)//2=2` 的上限，无解，返回空串。
- **视频覆盖**：片段 `[[0,2],[1,5],[4,8]]`，要覆盖 `[0,8]`。选 `[1,5]`、`[4,8]` 再配一个能从 0 接上的……实际上最优选法是 `[0,2]`、`[1,5]`、`[4,8]` 三段，答案是 **3**。

本章把三个解法并排放，就是让你看清"机制组合"时接口怎么定、正确性怎么各证各的。

## 2. 基础知识：先读懂这些词

- **机制组合与接口**：把一道题拆成两个你已掌握的机制（例如"Counter 计数"和"heapq 堆"），接口指两者之间传递的数据及其含义（比如"堆里放 (剩余次数, 标签)"）。定义清楚接口，组合才可推理、可测试。
- **局部选择 + 结构约束**：三道题的公共形状——每一步都在"当前可选的局部"里挑一个（局部选择），同时受全局规则限制（同任务间隔 n、相邻不同字符、片段必须首尾相接）。识别这个形状有用，但**不能**直接推出统一贪心。
- **下界论证（两个下界取最大）**：证明"至少需要这么长"的办法。冷却问题有两个显然下界：总任务数 `len(tasks)`（每步最多做一个任务），以及最高频任务撑起的框架 `(f-1)(n+1)+c`。最优值等于两者的最大值，前提是下界可以达到。
- **计数公式 vs 事件模拟**：同一个问题的两种实现路线。公式 `max(len(tasks),(maximum-1)*(n+1)+tied)` 是 O(m) 的闭式计算；`cooldown_schedule` 则逐步模拟时间线、真的把空闲槽排出来。两条路线互相核验，正是练习 1 的主题。
- **堆（优先队列）**：`heapq` 每次弹出"最优先"的元素。冷却和重排里都用它实现"每次先消耗剩余次数最多的字符"——Python 的堆是小顶堆，所以存负次数让最多的排最前。
- **暂扣机制（held）**：重排字符串时，刚输出过的字符先扣在手里不上堆，下一轮再放回去。它是"禁止相邻重复"这条结构约束在堆接口上的落点。
- **覆盖前沿（covered）**：视频覆盖贪心的状态量——"目前已经连续覆盖到哪里"。每一步都在"起点能接上前沿"的片段里选"终点最远的"，把前沿推得尽量远；片段的输入顺序无关，因为先排了序。
- **交换论证**：证明区间贪心正确的标准工具：任何解里的一个片段都可以换成"同样能接上、但结束更远"的片段，段数不增、后续可达范围不缩。

## 3. 思路推导：从小例子开始

- **Step 1（冷却）：先找两个下界。** 设 f 是最高频任务的出现次数。下界一：`len(tasks)`，因为每个时间槽最多执行一个任务。下界二：把最高频任务 A 的前 f-1 次 executions 每次后面隔出 n 个槽的框架，A 自己就占了 `(f-1)(n+1)` 个位置；若最高频任务有 c 个并列，它们的"最后一轮"还要挤进同一列，框架变成 `(f-1)(n+1)+c`。手算 `AAABBB`、n=2：f=3、c=2，框架 = `2×3+2 = 8`；总任务数 = 6；答案下界 = `max(6,8) = 8`。
- **Step 2（冷却）：说明下界可以达到。** 框架里的空位先填其他任务；如果任务多到填满还溢出（下界一占上风），说明根本不需要空闲槽。所以答案就是公式值。**接口注意**：公式只回答"多长"，不回答"每格放谁"——这正是要另配模拟器的原因。
- **Step 3（冷却）：用事件模拟核验。** `cooldown_schedule` 每个时刻先看冷却队列里谁到点了（`cooling[0][0]<=time`）就放回就绪堆，然后从就绪堆里弹"剩余最多"的执行；没有可执行的就把槽位记为 None。手推 `AAABBB`、n=2 的前 4 步：t=0 执行 A（A 冷却到 t=3）、t=1 执行 B（B 到 t=4）、t=2 双方都在冷却填 None、t=3 A 到点弹出执行（下次到 t=6）……完整时间线 `[A,B,None,A,B,None,A,B]`，长度 8，与公式一致——“运行示例”部分 第一张表就是这份逐槽 trace。
- **Step 4（重排）：先判可行性。** 出现最多的字符如果超过 `(m+1)//2`（m 是串长），间隙装不下它，直接返回空串。`aaab`：a 出现 3 次 > `(4+1)//2 = 2`，无解；`aaabbc`：a 出现 3 次 ≤ 3，可行。
- **Step 5（重排）：堆 + 暂扣，一轮一轮手算。** 初始堆 `[(-3,a),(-2,b),(-1,c)]`，手里 `held=(0,'')`。第 1 轮弹 a 输出，手里换成 a；第 2 轮弹 b 输出，把上一轮扣的 a 放回堆，手里换成 b；第 3 轮弹 a 输出、放回 b；第 4 轮弹 b 输出、放回 a；第 5 轮弹 a 输出（a 用完）；第 6 轮弹 c 输出。得到 `ababac`，相邻两两不同。堆空后检查手里那字符是否刚好用完（`held[0]==0`），没用完说明它被迫相邻，返回空串。
- **Step 6（覆盖）：排序后维护覆盖前沿。** 片段按左端点排序，`covered` 从 0 出发。手算 `[[0,2],[1,5],[4,8]]`、目标 8：前沿 0 时，能接上（`[0]<=0`）的只有 `[0,2]`，前沿推到 2，选 1 段；前沿 2 时 `[1,5]` 能接上，推到 5，选 2 段；前沿 5 时 `[4,8]` 能接上，推到 8，选 3 段；前沿到达 8，答案 3——“运行示例”部分 第二张表记录的正是这一结果。
- **Step 7（覆盖）：中途接不上就无解。** 若某一轮扫完所有能接上的片段，`farthest==covered`（前沿没动），说明出现断层，返回 -1（比如 `[[0,1],[2,3]]` 覆盖到 3：前沿 1 时没有片段左端 ≤1，直接 -1）。
- **Step 8：各证各的正确性，再互相对拍。** 冷却靠下界可达；重排靠"优先消耗最多者，避免把稀缺的高频字符留到最后被迫相邻"；覆盖靠交换论证。“自动测试”部分 再用 BFS 暴力冷却和子集枚举暴力覆盖做小规模对拍，公式、模拟、暴力三方一致才算数。

## 4. 核心代码：least_interval

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def least_interval(tasks, n):
    if n < 0:
        raise ValueError('Cooldown must be nonnegative')
    counts = Counter(tasks)
    if not counts:
        return 0
    maximum = max(counts.values())
    tied = sum((value == maximum for value in counts.values()))
    return max(len(tasks), (maximum - 1) * (n + 1) + tied)
```

### 逐段读懂代码

### 1. 冷却公式 `least_interval`

```python
def least_interval(tasks, n):
    if n < 0:
        raise ValueError('Cooldown must be nonnegative')
    counts = Counter(tasks)
    if not counts:
        return 0
    maximum = max(counts.values())
    tied = sum((value == maximum for value in counts.values()))
    return max(len(tasks), (maximum - 1) * (n + 1) + tied)
```

- 先做输入检查（n 非负）和空输入处理（无任务返回 0）。
- `maximum` 就是推导里的 f，`tied` 就是并列最高频的任务数 c，`sum(value==maximum ...)` 数了一遍布尔值。
- 返回 `max(len(tasks),(maximum-1)*(n+1)+tied)`，即两个下界的最大值。整个函数只数不排，O(m) 次计数就出答案。

### 2. 冷却模拟 `cooldown_schedule`

（对应程序见下方分段实现与完整代码。）

- 这是"事件模拟"一侧的接口：`ready` 堆存 (负剩余次数, 标签)——负号让剩余最多的最先弹出，标签参与比较保证元组可比（注释里"labels must be comparable"就是这个意思）。
- `cooling` 双端队列存 `(解冻时刻, 剩余负次数, 标签)`，按解冻时刻先进先出。每个 time 步，先批量把到点 (`<=time`) 的任务从队头搬回 ready 堆。
- 有就绪任务就弹一个执行并记 trace；`count+1<0` 说明还没用完，安排它 `time+n+1` 之后才能再次执行（本次执行 + 冷却 n 槓）。`count+1==0` 时任务用完，不再入队。
- 没有就绪任务时 `trace.append(None)` 记一个空闲槽。函数返回完整时间线，长度即总时长——它和公式解的是同一问题，构成互相验证的一对接口。

### 3. 字符串重排 `reorganize_string`

```python
def reorganize_string(s):
    counts = Counter(s)
    if max(counts.values(), default=0) > (len(s) + 1) // 2:
        return ''
    heap = [(-count, char) for char, count in counts.items()]
    heapq.heapify(heap)
    held = (0, '')
    out = []
    while heap:
        count, char = heapq.heappop(heap)
        out.append(char)
        if held[0] < 0:
            heapq.heappush(heap, held)
        held = (count + 1, char)
    return ''.join(out) if held[0] == 0 else ''
```

- 第一行判可行性：最高频字符超过 `(len(s)+1)//2` 时间隙装不下，提前返回空串；`default=0` 顺手处理了空串输入。
- 核心是 `held` 这个"暂扣"槽：每轮弹出堆顶字符输出，然后把**上一轮**扣住的字符（还有剩余时）放回堆，再把手里的换成**这一轮**刚输出的字符。同一字符不可能连续两轮被弹出，"禁止相邻重复"这条约束就藏在这一次"延迟入堆"里。
- 循环结束（堆空）后，手里若还剩未用完的字符（`held[0]<0`），说明它最终被迫相邻，返回 `''`；否则把输出列表拼成字符串返回。

### 4. 视频覆盖 `video_stitching`

```python
def video_stitching(clips, time):
    if time < 0:
        raise ValueError('Nonnegative target time required')
    clips = sorted(clips)
    i = answer = 0
    covered = 0
    while covered < time:
        farthest = covered
        while i < len(clips) and clips[i][0] <= covered:
            farthest = max(farthest, clips[i][1])
            i += 1
        if farthest == covered:
            return -1
        covered = farthest
        answer += 1
    return answer
```

- 排序把片段按左端点从左到右排好；`i` 是扫描指针，`covered` 是覆盖前沿。**接口**：排序后的 `clips` 只被单向扫描一遍，指针从不回退。
- 外层 `while covered<time`：前沿没到目标就再选一段。内层 `while` 把所有"左端点能接上前沿"的片段都看一遍，取它们右端点的最大值 `farthest`——这就是"在可接上的片段里选结束最远的"。
- `if farthest==covered:return -1`：能接上的片段一个都没有（或都接不上前沿之外），出现断层，无解。
- `covered=farthest;answer+=1` 推进前沿、计数。目标为 0 时循环一次都不进，直接返回 0。

## 5. 分段实现：按顺序运行

**本章接口：** `least_interval(tasks, n)`、`reorganize_string(s)`、`video_stitching(clips, time)`

### 导入本章使用的库

In [1]:
from collections import Counter, deque
import heapq

### least_interval

In [2]:
def least_interval(tasks, n):
    if n < 0:
        raise ValueError('Cooldown must be nonnegative')
    counts = Counter(tasks)
    if not counts:
        return 0
    maximum = max(counts.values())
    tied = sum((value == maximum for value in counts.values()))
    return max(len(tasks), (maximum - 1) * (n + 1) + tied)

### cooldown_schedule

In [3]:
def cooldown_schedule(tasks, n):
    """Small-case trace with None for idle slots; labels must be comparable."""
    if n < 0:
        raise ValueError('Cooldown must be nonnegative')
    ready = [(-count, label) for label, count in Counter(tasks).items()]
    heapq.heapify(ready)
    cooling = deque()
    trace = []
    time = 0
    while ready or cooling:
        while cooling and cooling[0][0] <= time:
            _, count, label = cooling.popleft()
            heapq.heappush(ready, (count, label))
        if ready:
            count, label = heapq.heappop(ready)
            trace.append(label)
            if count + 1 < 0:
                cooling.append((time + n + 1, count + 1, label))
        else:
            trace.append(None)
        time += 1
    return trace

### reorganize_string

In [4]:
def reorganize_string(s):
    counts = Counter(s)
    if max(counts.values(), default=0) > (len(s) + 1) // 2:
        return ''
    heap = [(-count, char) for char, count in counts.items()]
    heapq.heapify(heap)
    held = (0, '')
    out = []
    while heap:
        count, char = heapq.heappop(heap)
        out.append(char)
        if held[0] < 0:
            heapq.heappush(heap, held)
        held = (count + 1, char)
    return ''.join(out) if held[0] == 0 else ''

### video_stitching

In [5]:
def video_stitching(clips, time):
    if time < 0:
        raise ValueError('Nonnegative target time required')
    clips = sorted(clips)
    i = answer = 0
    covered = 0
    while covered < time:
        farthest = covered
        while i < len(clips) and clips[i][0] <= covered:
            farthest = max(farthest, clips[i][1])
            i += 1
        if farthest == covered:
            return -1
        covered = farthest
        answer += 1
    return answer

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [6]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

trace=cooldown_schedule('AAABBB',2)
show_table(['time','task'],list(enumerate(trace)))
show_table(['problem','input','result'],[
 ['reorganization','aaabbc',reorganize_string('aaabbc')],
 ['coverage',[[0,2],[1,5],[4,8]],video_stitching([[0,2],[1,5],[4,8]],8)]])

time,task
0,A
1,B
2,None
3,A
4,B
5,None
6,A
7,B


problem,input,result
reorganization,aaabbc,ababac
coverage,"[[0, 2], [1, 5], [4, 8]]",3


## 7. 正确性、适用条件与复杂度

冷却框架给出必要下界：最后一轮之前，每个最高频任务之间都要留足间隔；其他任务可填入空位，若超出空位则无须空闲。重排中最大频率不超过其余字符间隙数是可行必要条件，优先消耗最多者避免把稀缺间隙留到最后。区间贪心用交换论证：用结束更远、同样可接上的片段替换任一首段，不增加段数且不缩短后续可达范围。

**代价：** 冷却最短长度计算 O(m)，不同标签空间 O(u)；演示时间线可能包含大量空闲，仅用于小输入。重排 O(m log u)，覆盖排序 O(k log k)、扫描 O(k)。

### 展开理解

**冷却公式为什么对。** 框架 `(f-1)(n+1)+c` 是必要下界：最后一轮之前，最高频任务每两次执行之间必须隔满 n 个槽，这 f-1 个"间隔块"是跑不掉的；并列最高频的 c 个任务还要在框架末尾各占一个位置。同时其他任务总能填进框架空位，如果任务多到溢出，那说明"总任务数"这个下界更大、空闲根本不需要——两个下界的最大值总能被某个真实排期达到。模拟器则从另一头核验：它构造的真实时间线长度恰好等于公式值，且每两个同类任务的实际间隔都 > n（测试里逐槽断言了这一点）。

**重排为什么对。** 可行性条件（最高频 ≤ (m+1)//2）是必要的：m 个字符只有 m-1 个相邻间隙，最高频字符需要至少"自己次数-1"个间隙隔开它。贪心"每轮优先消耗剩余最多的、同时扣住刚输出的"是安全的：如果不优先消耗最多者，稀缺的高频字符会被留到结尾，那时其他字符已经用光，它们只能彼此相邻——提前消耗避免了这种绝境。堆空后手里还有剩，等价于可行性条件被违反，两种检查殊途同归。

**覆盖为什么对。** 交换论证：考虑任何一个最优解，把它的第一段换成"同样能从 0 接上、但结束最远的片段"，覆盖范围不减、段数不变；对后续每一步做同样的替换，就把它逐段改造成贪心解。所以贪心不会比最优差；而贪心自己就是合法解，于是它就是最优。

**代价。** 冷却公式是 O(m)（m 为任务总数）、去重后标签空间 O(u)；注意模拟器的时间线可能含大量空闲槽（比如只有一个任务 A 出现 3 次、n=100），所以它只用于小输入演示。重排每个输出字符一次堆操作，O(m log u)。覆盖先排序 O(k log k)、再单向扫描 O(k)（k 为片段数）。拿具体规模说：任务一万个、不同标签 26 个，冷却公式只数一遍；片段一千个，排序一千 log 一千、扫描一千步出答案。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分四组：

1. **冒烟与边界**：`least_interval('AAABBB',2)==8`（正常值，手推的公式例子）；`reorganize_string('aaab')==''`（不可行值，a 超过间隙上限）；`video_stitching([[0,1],[2,3]],3)==-1`（覆盖到 1 后出现断层，测无解分支）；`video_stitching([],0)==0`（边界：目标为 0 时零片段即可覆盖）。
2. **冷却：BFS 暴力对拍 + 时间线核验**。`brute_cooldown` 用 BFS 枚举（剩余次数向量, 各任务冷却计数）的状态空间，每步可执行任一就绪任务或空转，返回最少步数——它是独立的正确性参照。对三个标签、每个出现 0..2 次的全部组合 × n=0,1,2，断言公式值等于 BFS 值。此外还把 `cooldown_schedule` 的 trace 拿来验三件事：长度等于最优值、非空槽的字符多重集与输入完全一致（`Counter(x for x in trace if x is not None)==Counter(tasks)`）、同类任务两次出现的间隔严格大于 n。**这一组同时测了"公式"和"模拟"两个接口的一致性。**
3. **重排：穷举验证**。对上面同一批小输入，先用频率条件判断"可能可行"，可行时断言输出字符多重集等于输入（不许丢字、造字）且相邻字符两两不同；不可行时断言输出恰为空串。多重集比较就是防"输出对了形状但字符不对"的接口检查。
4. **覆盖：随机对拍**。`brute_cover` 按段数从少到多枚举所有子集、检查排序后能否首尾相接地覆盖目标——是指数级但小规模可跑的参照。用固定种子 `Random(172)` 生成 250 组随机片段（每组长 0..7、左端 0..6、长度 1..4）和随机目标 0..9，断言贪心答案与暴力答案逐一相等，顺带覆盖了 -1 的各种断层情形。

In [7]:
from itertools import product, combinations

from random import Random

assert least_interval('AAABBB', 2) == 8

assert reorganize_string('aaab') == ''

assert video_stitching([[0, 1], [2, 3]], 3) == -1

assert video_stitching([], 0) == 0

def brute_cooldown(counts, n):
    start = (tuple(counts), (0,) * len(counts))
    queue = deque([(start, 0)])
    seen = {start}
    while queue:
        (remaining, wait), time = queue.popleft()
        if not any(remaining):
            return time
        choices = [i for i in range(len(counts)) if remaining[i] and wait[i] == 0] + [None]
        for choice in choices:
            new_counts = list(remaining)
            new_wait = [max(0, x - 1) for x in wait]
            if choice is not None:
                new_counts[choice] -= 1
                new_wait[choice] = n
            state = (tuple(new_counts), tuple(new_wait))
            if state not in seen:
                seen.add(state)
                queue.append((state, time + 1))

for counts in product(range(3), repeat=3):
    tasks = ''.join((chr(65 + i) * count for i, count in enumerate(counts)))
    for n in range(3):
        expected = brute_cooldown(counts, n)
        assert least_interval(tasks, n) == expected
        trace = cooldown_schedule(tasks, n)
        assert len(trace) == expected and Counter((x for x in trace if x is not None)) == Counter(tasks)
        last = {}
        for time, label in enumerate(trace):
            if label is not None:
                assert label not in last or time - last[label] > n
                last[label] = time
    out = reorganize_string(tasks)
    possible = max(counts, default=0) <= (len(tasks) + 1) // 2
    if possible:
        assert Counter(out) == Counter(tasks) and all((a != b for a, b in zip(out, out[1:])))
    else:
        assert out == ''

def brute_cover(clips, target):
    for count in range(len(clips) + 1):
        for subset in combinations(clips, count):
            end = 0
            for left, right in sorted(subset):
                if left > end:
                    break
                end = max(end, right)
            if end >= target:
                return count
    return -1

rng = Random(172)

for _ in range(250):
    clips = []
    for i in range(rng.randrange(8)):
        left = rng.randrange(7)
        clips.append((left, left + rng.randrange(1, 5)))
    target = rng.randrange(10)
    assert video_stitching(clips, target) == brute_cover(clips, target)

print('N172: 所有本章断言通过')

N172: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 同题实现计数公式与事件模拟。

**练习 2：** 解释正确组合而非无依据堆叠模板。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（同题双实现）**：任选本章一题，把"计数公式"和"事件模拟"两种路线都写出来，再用第三个暴力实现当裁判。以冷却为例，输入输出和边界要写清楚（n 为负要抛错、空任务返回 0）；手算示例直接用 `AAABBB`、n=2：公式给 8，模拟时间线 `[A,B,None,A,B,None,A,B]` 也是 8，三方一致。重点是体会两种接口各自回答什么：公式答"多长"，模拟答"每格放谁"。
- **练习 2（解释正确组合）**：挑一个"看似能套同一个贪心模板"的题对（比如本章的重排和覆盖），分别写出它们"局部选择"的依据和结构约束的落点：重排的约束落在"暂扣延迟入堆"上，覆盖的约束落在"片段必须接上前沿"上。然后构造一个小输入，展示"把 A 题的贪心直接搬到 B 题"会在哪一步失败（例如把"选剩余最多"用在覆盖上，会优先选长片段而不是接得上的片段）。写明每道题输入输出与边界，用手算例子展示失败的那一步，再用本章接口或暴力解验证你的分析。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [8]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter, deque

import heapq

def least_interval(tasks, n):
    if n < 0:
        raise ValueError('Cooldown must be nonnegative')
    counts = Counter(tasks)
    if not counts:
        return 0
    maximum = max(counts.values())
    tied = sum((value == maximum for value in counts.values()))
    return max(len(tasks), (maximum - 1) * (n + 1) + tied)

def cooldown_schedule(tasks, n):
    """Small-case trace with None for idle slots; labels must be comparable."""
    if n < 0:
        raise ValueError('Cooldown must be nonnegative')
    ready = [(-count, label) for label, count in Counter(tasks).items()]
    heapq.heapify(ready)
    cooling = deque()
    trace = []
    time = 0
    while ready or cooling:
        while cooling and cooling[0][0] <= time:
            _, count, label = cooling.popleft()
            heapq.heappush(ready, (count, label))
        if ready:
            count, label = heapq.heappop(ready)
            trace.append(label)
            if count + 1 < 0:
                cooling.append((time + n + 1, count + 1, label))
        else:
            trace.append(None)
        time += 1
    return trace

def reorganize_string(s):
    counts = Counter(s)
    if max(counts.values(), default=0) > (len(s) + 1) // 2:
        return ''
    heap = [(-count, char) for char, count in counts.items()]
    heapq.heapify(heap)
    held = (0, '')
    out = []
    while heap:
        count, char = heapq.heappop(heap)
        out.append(char)
        if held[0] < 0:
            heapq.heappush(heap, held)
        held = (count + 1, char)
    return ''.join(out) if held[0] == 0 else ''

def video_stitching(clips, time):
    if time < 0:
        raise ValueError('Nonnegative target time required')
    clips = sorted(clips)
    i = answer = 0
    covered = 0
    while covered < time:
        farthest = covered
        while i < len(clips) and clips[i][0] <= covered:
            farthest = max(farthest, clips[i][1])
            i += 1
        if farthest == covered:
            return -1
        covered = farthest
        answer += 1
    return answer

# 示例与回归测试
from itertools import product, combinations

from random import Random

assert least_interval('AAABBB', 2) == 8

assert reorganize_string('aaab') == ''

assert video_stitching([[0, 1], [2, 3]], 3) == -1

assert video_stitching([], 0) == 0

def brute_cooldown(counts, n):
    start = (tuple(counts), (0,) * len(counts))
    queue = deque([(start, 0)])
    seen = {start}
    while queue:
        (remaining, wait), time = queue.popleft()
        if not any(remaining):
            return time
        choices = [i for i in range(len(counts)) if remaining[i] and wait[i] == 0] + [None]
        for choice in choices:
            new_counts = list(remaining)
            new_wait = [max(0, x - 1) for x in wait]
            if choice is not None:
                new_counts[choice] -= 1
                new_wait[choice] = n
            state = (tuple(new_counts), tuple(new_wait))
            if state not in seen:
                seen.add(state)
                queue.append((state, time + 1))

for counts in product(range(3), repeat=3):
    tasks = ''.join((chr(65 + i) * count for i, count in enumerate(counts)))
    for n in range(3):
        expected = brute_cooldown(counts, n)
        assert least_interval(tasks, n) == expected
        trace = cooldown_schedule(tasks, n)
        assert len(trace) == expected and Counter((x for x in trace if x is not None)) == Counter(tasks)
        last = {}
        for time, label in enumerate(trace):
            if label is not None:
                assert label not in last or time - last[label] > n
                last[label] = time
    out = reorganize_string(tasks)
    possible = max(counts, default=0) <= (len(tasks) + 1) // 2
    if possible:
        assert Counter(out) == Counter(tasks) and all((a != b for a, b in zip(out, out[1:])))
    else:
        assert out == ''

def brute_cover(clips, target):
    for count in range(len(clips) + 1):
        for subset in combinations(clips, count):
            end = 0
            for left, right in sorted(subset):
                if left > end:
                    break
                end = max(end, right)
            if end >= target:
                return count
    return -1

rng = Random(172)

for _ in range(250):
    clips = []
    for i in range(rng.randrange(8)):
        left = rng.randrange(7)
        clips.append((left, left + rng.randrange(1, 5)))
    target = rng.randrange(10)
    assert video_stitching(clips, target) == brute_cover(clips, target)

print('N172: 所有本章断言通过')

N172: 所有本章断言通过


## 11. 代表题与迁移

- **621. Task Scheduler（canonical）**：练"双下界取最大 + 公式/模拟双实现互相核验"，`least_interval` 与 `cooldown_schedule` 就是它的两条解题路线。
- **767. Reorganize String（canonical）**：练"堆 + 暂扣机制"的组合，重点是可行性条件 `(m+1)//2` 和"优先消耗最多者"的论证。
- **1024. Video Stitching（canonical）**：练"排序 + 覆盖前沿 + 交换论证"的区间贪心组合，`video_stitching` 就是其完整教学实现。

notebook 结尾照例提醒：这些题号用于知识映射，本章实现遵循教学契约，不要把教学实例当成官方题的完整答案。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。